### Vorverarbeitung der Daten

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data.csv")
df["date"] = pd.to_datetime(df["DateTime"])

# Tarife werden vorerst zusammengelegt
df = df.groupby("date", as_index=False).agg({"KWH": "sum", "meters": "sum"}).reset_index()

# Normalisierung: Stromverbrauch eines Verbrauchers für jeden Zeitpunkt
df["kwh"] = df["KWH"] / df["meters"]

df = df[["date", "KWH", "kwh", "meters"]]
display(df)

# Erste Visualisierung des Datensatzes als täglicher Durchschnittsverbrauch
series = df.set_index("date")["kwh"].sort_index()
series = series.resample("D").mean()
fig, ax = plt.subplots(figsize=(15, 6))
series.plot(ax=ax, color="#96002A")
ax.set(
    title="Täglicher durchschnittlicher Stromverbrauch",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
plt.show()

### Datenbereinigung

Im Datensatz sind fehlerhaft Datenpunkte enthalten, bei denen kein Verbraucher geschaltet ist und die Messungen außerhalb dem 30-Minuten-Taktes liegen.

In [ ]:
# Prüfung auf Datenpunkte mit keinem Verbrauch
display(df.loc[(df["KWH"] == 0)])

# Falls die Prüfungen nichts zurücklieferen, können die fehlerhaften Datenpunkte entfernt werden
# Prüfung auf korrekte Datenpunkte mit keinem Verbrauch
display(df.loc[(df["KWH"] == 0) & ((df["date"].dt.minute % 30 == 0) & (df["date"].dt.second == 0))])
# Prüfung auf korrekte Datenpunkte außerhalb des 30-Minuten-Taktes
display(df.loc[~(df["KWH"] == 0) & ~((df["date"].dt.minute % 30 == 0) & (df["date"].dt.second == 0))])

# Durch .asfreq() fallen die fehlerhaften Datenpunkte weg
df = df.set_index("date")[["kwh"]].sort_index().asfreq("30min")
display(df["kwh"].isna().sum())

### Untersuchung der Zeitreihe

In [ ]:
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

x = mdates.date2num(df.index)
p = np.poly1d(np.polyfit(x, df["kwh"], 1))
# df["kwh_trend"] = p(x)
df["kwh_trend"] = df["kwh"].rolling(window=336, center=True).mean()  # Mit (wöchentlicher) Glättung

df["kwh_trendbereinigt"] = df["kwh"] - df["kwh_trend"]
ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.7, label="Verbrauch")
ax.plot(df.index, df["kwh_trendbereinigt"], color="#F3BE2B", alpha=0.7, label="Verbrauch trendbereinigt")
ax.plot(df.index, df["kwh_trend"], color="#000000", alpha=1, label="Trend")
ax.set(
    title="Stromverbrauch eines Verbrauchers",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(axis="y", alpha=0.5)
plt.show()

In [ ]:
# .hour, .dayofweek, .day
df["focus"] = df.index.hour * 2 + df.index.minute // 30  # 48 Zeitpunkte pro Tag

seasonal = df.groupby("focus")["kwh_trendbereinigt"].mean()

fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

seasonal.plot(ax=ax, color="#D386FF", alpha=1, label="Verbrauch")
ax.set(
    title="Saisonalität (Fokus)",
    xlabel="Fokus",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(axis="y", alpha=0.5)
plt.show()

df = pd.merge(df, seasonal, on="focus", suffixes=("", "_focus"))
df

In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

df["kwh_bereinigt"] = df["kwh_trendbereinigt"] - df["kwh_trendbereinigt_focus"]
ax.plot(df.index, df["kwh"], color="#6B001E", alpha=0.5, label="Verbrauch")
ax.plot(df.index, df["kwh_trendbereinigt"], color="#F3BE2B", alpha=0.5, label="Verbrauch trendbereinigt")
ax.plot(df.index, df["kwh_bereinigt"], color="#9B0000", alpha=0.7, label="Verbrauch bereinigt")
ax.plot(df.index, df["kwh_trend"], color="#000000FF", alpha=0.7, label="Trend")
ax.plot(df.index, df["kwh_trendbereinigt_focus"], color="#D386FF", alpha=0.7, label="Saisonalität")
ax.set(
    title="Stromverbrauch eines Verbrauchers",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.set_xlim(pd.Series(df.index).quantile(0.49), pd.Series(df.index).quantile(0.52))  # Fokus/Zoom
ax.legend()
ax.grid(axis="y", alpha=0.5)
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import STL

decomposition = STL(
    df["kwh"],
    period=48,  # 48 = Tag, 336 = Woche
)
decomposition = decomposition.fit()
_ = decomposition.plot()

In [ ]:
# from statsmodels.tsa.seasonal import seasonal_decompose

# decomposition = seasonal_decompose(df["kwh"], model="additive", period=48)
# _ = decomposition.plot()

In [ ]:
trend = decomposition.trend
seasonal = decomposition.seasonal
resid = decomposition.resid

fig, ax = plt.subplots(4, 1, figsize=(15, 6), sharex=True)

for p in ax:
    p.set_xlim(pd.Series(df.index).quantile(0.49), pd.Series(df.index).quantile(0.52))

ax[0].plot(df["kwh"], color="#6B001E")
ax[0].set_title("Stromverbrauch")

ax[1].plot(trend, color="#000000FF")
ax[1].set_title("Trend")

ax[2].plot(seasonal, color="#D386FF", alpha=0.7)
ax[2].set_title("Saisonalität")

ax[3].plot(resid, color="#9B0000")
ax[3].set_title("Residuum")

plt.show()

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

print("kwh_bereinigt:")
_ = plot_acf(df["kwh_bereinigt"].dropna(), lags=48)
_ = plot_pacf(df["kwh_bereinigt"].dropna(), lags=48)
plt.show()
print("statsmodels STL (Season-Trend decomposition using LOESS):")
_ = plot_acf(resid.dropna(), lags=48)
_ = plot_pacf(resid.dropna(), lags=48)
plt.show()

### Vorhersage

In [ ]:
from sklearn.linear_model import LinearRegression

df_model = df.copy()
df_model["kwh_bereinigt_t-1"] = df_model.kwh_bereinigt.shift(1)
df_model["kwh_bereinigt_t-2"] = df_model.kwh_bereinigt.shift(2)
df_model["kwh_bereinigt_t-3"] = df_model.kwh_bereinigt.shift(3)
df_model = df_model.dropna(
    subset=["kwh_bereinigt", "kwh_bereinigt_t-1", "kwh_bereinigt_t-2", "kwh_bereinigt_t-3"]
).copy()

X = df_model[["kwh_bereinigt_t-1", "kwh_bereinigt_t-2", "kwh_bereinigt_t-3"]]
y = df_model["kwh_bereinigt"]

model = LinearRegression()
model.fit(X, y)

df_model["kwh_vorhersage"] = model.predict(X)

df_prediction = pd.merge(df, df_model["kwh_vorhersage"], left_index=True, right_index=True)

print("Regression Coefficients:", model.coef_)
print("Regression Intercept:", model.intercept_)
df_prediction.tail(40)

In [ ]:
from sklearn.linear_model import LinearRegression

df_model = df.copy()

df_model["target"] = df_model.kwh_bereinigt.shift(-12)
df_model["kwh_bereinigt_t-48"] = df_model.kwh_bereinigt.shift(48)

for lag in range(1, 13):
    df_model[f"kwh_bereinigt_t-{lag}"] = df_model.kwh_bereinigt.shift(lag)

lags = [f"kwh_bereinigt_t-{lag}" for lag in range(1, 13)] + ["kwh_bereinigt_t-48"]

df_model = df_model.dropna()

X = df_model[lags]
y = df_model["target"]

model = LinearRegression()
model.fit(X, y)

df_model["kwh_vorhersage"] = model.predict(X)

df_model["kwh_vorhersage"] = df_model["kwh_vorhersage"].shift(12)

df_prediction = pd.merge(df, df_model["kwh_vorhersage"], left_index=True, right_index=True)

print("Regression Coefficients:", model.coef_)
print("Regression Intercept:", model.intercept_)
df_prediction.tail(40)

In [ ]:
plt.figure(figsize=(15, 6))
plt.plot(df_prediction["kwh_bereinigt"], alpha=0.7, color="red", label="Bereinigt")
plt.plot(df_prediction["kwh_vorhersage"], alpha=0.7, color="blue", label="Vorhersage")
plt.legend()

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(df["kwh"], order=(2, 1, 2)).fit()
forecast = model.forecast(steps=12)

fig, ax = plt.subplots(figsize=(15, 6))
ax.margins(x=0)

ax.plot(forecast.index, forecast, color="#A5002F75", label="6-Stunden-Vorhersage")
df["kwh"].tail(50).plot(ax=ax, color="#440013", label="Verbrauch")
ax.set(
    title="",
    xlabel="Datum",
    ylabel="Kilowattstunde",
)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.show()

### Varianzen

In [ ]:
df_var = df.copy()
df_var["trend"] = decomposition.trend
df_var["seasonal"] = decomposition.seasonal
df_var["resid"] = decomposition.resid

varianzen = df_var[["kwh", "trend", "seasonal", "resid"]].var()
varianzen["ryth"] = decomposition.seasonal.var()
anteile = varianzen / varianzen["kwh"]

print(
    f"Von der Varianz der Zeitreihe werden:\n\t{(anteile['trend'] * 100):2.1f}% durch den Trend erklärt"
    f"\n\t{(anteile['ryth'] * 100):2.1f}% durch die Periodizität erklärt"
    f"\n\t{(anteile['resid'] * 100):2.1f}% haben andere Ursachen (Residuum)"
)